<a href="https://colab.research.google.com/github/codewithUswaFatima/urdu-ocr-codesaviours-si26-Uswa/blob/main/SI26_Uswa_Week4_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install Libraries

In [4]:
# Install compatible versions in ONE go (avoids version conflicts later)
!pip install -q -U transformers==4.46.3 tokenizers==0.20.3 sentencepiece protobuf accelerate datasets huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 11.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires pro

### ⚠️ IMPORTANT: Restart the runtime now
After running the install cell above, go to **Runtime → Restart session** (Colab) or **Kernel → Restart Kernel** (Jupyter), then run all cells again **starting from Step 2**.

This is required because Python caches which optional libraries (like `sentencepiece`) were available *at the moment `transformers` was first imported*. Installing them afterward in the same session does not update that cache — that's exactly why you were getting:
`ValueError: Couldn't instantiate the backend tokenizer ... You need to have sentencepiece or tiktoken installed ...`
even though sentencepiece was installed later in the notebook.


# Step 2: Import Libraries

In [5]:
import torch
import pandas as pd
from transformers import (
    TrOCRProcessor,
    VisionEncoderDecoderModel
)
from torch.utils.data import Dataset, DataLoader
from PIL import Image

# Step 3: Load Dataset

In [6]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [7]:
CSV_FILE = "/content/gdrive/MyDrive/labels (2).csv"
IMAGE_FOLDER = "/content/data/processed"

In [8]:
df = pd.read_csv(CSV_FILE)
df.head()

,image,text
0,raw/dataset/0.png,پشاور،بنوں (نمائندہ جنگ،اے ایف پی) بنوں میں اق...
1,raw/dataset/1.png,اسکے ساتھ ملحقہ علاقے کے عوام نے بجلی و گیس \n
2,raw/dataset/10.png,نے بجلی وگیس کی لوڈشیڈنگ کیخلاف مظاہرہ کیا۔ مظ...
3,raw/dataset/100.png,نہیں آ یا۔ آخریہ کونسا مذہب ہے؟کیا اسلام کے نا...
4,raw/dataset/1000.png,ضرور ہے۔ انہوں نے کہا کہ میرا احمدیوں سے کوئی \n


# Step 4: Load Processor

In [9]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
from transformers import TrOCRProcessor,VisionEncoderDecoderModel

processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/224 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

# Step 5: Create Dataset Class

In [10]:
class UrduOCRDataset(Dataset):

  def __init__(self,dataframe,image_folder,processor):
    self.dataframe = dataframe
    self.image_folder = image_folder
    self.processor = processor

  def __len__(self):
    return len(self.df)

  def __getitem__(self,idx):
    row = self.df.iloc[idx]
    image_path = f"{self.image_folder}/{row['file_name']}"
    image = Image.open(image_path).convert("RGB")
    pixel_values = self.processor(images = image, return_tensors = "pt").pixel_values.squeeze()

    labels = self.processor.tokenizer(
        row["text"],
        padding = "max_length",
        max_length = 128,
        trunication = True,
    ).input_ids()

    labels = [
        label if label!=processor.tokenizer.pad_token_id else -100 for label in labels
    ]

    return {
        "pixel_values": pixel_values,
        "labels": torch.tensor(labels)
    }


# Step 6: Train/Test Split

In [11]:
from sklearn.model_selection import train_test_split
train_df,test_df = train_test_split(
    df,
    test_size = 0.2,
    random_state= 42
)

# Step 7: Create Dataset Objects

In [12]:
train_dataset = UrduOCRDataset(train_df,IMAGE_FOLDER,processor)
test_dataset = UrduOCRDataset(test_df,IMAGE_FOLDER,processor)

# Step 8: Load TrOCR Model

In [14]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device}: device")
if device == "cpu":
  print("Warning No GPU Detected.")

Using cuda: device


In [15]:
processor = TrOCRProcessor.from_pretrained(
    "microsoft/trocr-base-printed"
)
model = VisionEncoderDecoderModel.from_pretrained(
    "microsoft/trocr-base-printed"
)
model.to(device)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.models.trocr.modeling_trocr.TrOCRForCausalLM'> is overwritten by shared decoder config: TrOCRConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_cross_attention": true,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classifier_dropout": 0.0,
  "cross_attention_hidden_size": 768,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder

generation_config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

VisionEncoderDecoderModel(
  (encoder): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTSdpaAttention(
            (attention): ViTSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=False)
              (key): Linear(in_features=768, out_features=768, bias=False)
              (value): Linear(in_features=768, out_features=768, bias=False)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linea

# Step 9: Configure Model

In [16]:
model.config.decode_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.vocab_size = model.config.decoder.vocab_size
print("Model loaded successfully!")

Model loaded successfully!
